# 04_04: Image Editing & Inpainting

**Objective:** Learn to edit existing images using text guidance — inpainting (filling in masked regions), image-to-image translation, and instruction-based editing.

**Prerequisites:** Notebook 04_03 (Text-to-Image). GPU strongly recommended.

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min VRAM | Recommended Setup | Notes |
|-------------|------------|------|----------|-------------------|-------|
| **Inpainting** | runwayml/stable-diffusion-inpainting | ~5GB | 6GB VRAM | RTX 4080+ | Fill masked regions |
| **Img2Img** | runwayml/stable-diffusion-v1-5 | ~5GB | 6GB VRAM | RTX 4080+ | Transform images |
| **Instruct** | timbrooks/instruct-pix2pix | ~5GB | 6GB VRAM | RTX 4080+ | Edit with instructions |

> **Note:** This notebook requires GPU. CPU fallbacks are provided for understanding the concepts but actual generation requires a CUDA-capable GPU.

## Expected Behaviors

### First Time Running
- **Model downloads**: ~5GB per model (cached after first download)
- Loading multiple models may need 10-16GB VRAM

### What You'll See
- **Inpainting**: Masked regions are filled with content matching the text prompt
- **Img2Img**: Entire images are transformed while preserving structure
- **InstructPix2Pix**: Natural language instructions modify the image

### Common Observations
- Inpainting blends seamlessly when the mask has soft edges
- Higher `strength` in img2img produces more dramatic changes
- InstructPix2Pix works best with clear, simple instructions

## Overview

This notebook covers three image editing techniques that build on the Stable Diffusion foundation from notebook 04_03:

| Technique | Input | Output | Example |
|-----------|-------|--------|--------|
| **Inpainting** | Image + mask + prompt | Image with masked area replaced | Remove object, change background |
| **Img2Img** | Image + prompt + strength | Transformed image | Style transfer, scene modification |
| **InstructPix2Pix** | Image + instruction | Edited image | "Make it winter", "Add sunglasses" |

### How Inpainting Works

```
Original Image → Encode to latent
Mask            → Mark regions to regenerate
Text Prompt     → Guide what to fill in
                        ↓
    Denoise ONLY the masked region (keep unmasked pixels)
                        ↓
    Decode → Seamlessly edited image
```

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Check for diffusers
try:
    import diffusers
    from diffusers import (
        StableDiffusionInpaintPipeline,
        StableDiffusionImg2ImgPipeline,
        StableDiffusionInstructPix2PixPipeline,
    )
    DIFFUSERS_AVAILABLE = True
    print(f'diffusers: {diffusers.__version__}')
except ImportError:
    DIFFUSERS_AVAILABLE = False
    print('diffusers not installed. Install with: pip install diffusers[torch]')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if device.type == 'cpu':
    print('\n⚠ WARNING: Image editing on CPU is extremely slow. GPU is required for practical use.')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Creating Test Images

We'll create synthetic test images so this notebook can run without external downloads. In practice, you'd use your own photos or images from the Hub.

In [ ]:
def create_test_image(
    width: int = 512,
    height: int = 512,
    scene: str = 'landscape',
) -> Image.Image:
    """Create a simple synthetic test image.

    Args:
        width: Image width.
        height: Image height.
        scene: Type of scene ('landscape', 'room', 'portrait').

    Returns:
        PIL Image.
    """
    image = Image.new('RGB', (width, height))
    draw = ImageDraw.Draw(image)
    
    if scene == 'landscape':
        # Sky
        draw.rectangle([0, 0, width, height // 2], fill=(135, 206, 235))
        # Ground
        draw.rectangle([0, height // 2, width, height], fill=(34, 139, 34))
        # Sun
        draw.ellipse([width - 120, 30, width - 40, 110], fill=(255, 223, 0))
        # Tree
        draw.rectangle([100, height // 2 - 80, 120, height // 2 + 10], fill=(101, 67, 33))
        draw.ellipse([60, height // 2 - 150, 160, height // 2 - 50], fill=(0, 100, 0))
    elif scene == 'room':
        # Walls
        draw.rectangle([0, 0, width, height], fill=(245, 222, 179))
        # Floor
        draw.rectangle([0, height * 2 // 3, width, height], fill=(160, 120, 80))
        # Window
        draw.rectangle([width // 3, height // 6, width * 2 // 3, height // 2], fill=(135, 206, 235))
        draw.rectangle([width // 3, height // 6, width * 2 // 3, height // 2], outline=(101, 67, 33), width=4)
    else:
        # Simple colored background
        draw.rectangle([0, 0, width, height], fill=(200, 200, 200))
        # Face placeholder
        draw.ellipse([width // 3, height // 4, width * 2 // 3, height * 3 // 4], fill=(255, 218, 185))
    
    return image


def create_mask(
    width: int = 512,
    height: int = 512,
    mask_region: tuple[int, int, int, int] = (150, 100, 350, 300),
) -> Image.Image:
    """Create a binary mask for inpainting.

    Args:
        width: Image width.
        height: Image height.
        mask_region: (left, top, right, bottom) of the white (inpaint) region.

    Returns:
        PIL Image mask (white = inpaint, black = keep).
    """
    mask = Image.new('RGB', (width, height), 'black')
    draw = ImageDraw.Draw(mask)
    draw.ellipse(mask_region, fill='white')
    return mask


# Create test images
test_image = create_test_image(512, 512, 'landscape')
test_mask = create_mask(512, 512, (150, 80, 350, 280))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(test_image)
axes[0].set_title('Test Image (Landscape)', fontsize=13)
axes[0].axis('off')
axes[1].imshow(test_mask)
axes[1].set_title('Inpainting Mask (white = edit)', fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Part 1: Inpainting

Inpainting fills in masked regions of an image with new content guided by a text prompt. The surrounding context is preserved, creating a seamless edit.

In [ ]:
INPAINT_MODEL = 'runwayml/stable-diffusion-inpainting'

if DIFFUSERS_AVAILABLE and device.type == 'cuda':
    print(f'Loading inpainting model: {INPAINT_MODEL}')
    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        INPAINT_MODEL,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(device)
    try:
        inpaint_pipe.enable_attention_slicing()
    except Exception:
        pass
    print('Inpainting model loaded.')
else:
    inpaint_pipe = None
    print('Inpainting requires diffusers + GPU. Showing concepts only.')

In [ ]:
def inpaint_image(
    image: Image.Image,
    mask: Image.Image,
    prompt: str,
    negative_prompt: str = 'blurry, low quality',
    num_steps: int = 25,
    guidance_scale: float = 7.5,
    seed: int = SEED,
) -> Image.Image | None:
    """Inpaint a masked region of an image.

    Args:
        image: Original PIL image.
        mask: Mask image (white = region to inpaint).
        prompt: Text description of what to fill in.
        negative_prompt: What to avoid.
        num_steps: Number of denoising steps.
        guidance_scale: Prompt guidance strength.
        seed: Random seed.

    Returns:
        Inpainted PIL Image, or None if unavailable.
    """
    if inpaint_pipe is None:
        print('Inpainting pipeline not available.')
        return None
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    result = inpaint_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )
    
    return result.images[0]


# Demonstrate inpainting with different prompts
inpaint_prompts = [
    'A beautiful red house with a chimney',
    'A lake reflecting the sky',
    'A field of sunflowers',
]

if inpaint_pipe is not None:
    fig, axes = plt.subplots(1, len(inpaint_prompts) + 1, figsize=(5 * (len(inpaint_prompts) + 1), 5))
    axes[0].imshow(test_image)
    axes[0].set_title('Original', fontsize=11)
    axes[0].axis('off')
    
    for idx, prompt in enumerate(inpaint_prompts):
        result = inpaint_image(test_image, test_mask, prompt)
        if result:
            axes[idx + 1].imshow(result)
            axes[idx + 1].set_title(prompt[:30] + '...', fontsize=9)
            axes[idx + 1].axis('off')
    
    plt.suptitle('Inpainting Results (same mask, different prompts)', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Inpainting examples (would generate with GPU):')
    for p in inpaint_prompts:
        print(f'  Prompt: "{p}"')

## Part 2: Image-to-Image Translation

Img2Img takes an existing image and transforms it according to a text prompt. The `strength` parameter controls how much the original image is modified:
- **strength=0.0**: No change (identical to input)
- **strength=0.5**: Moderate transformation (structure preserved, details changed)
- **strength=1.0**: Complete regeneration (only layout roughly preserved)

In [ ]:
if DIFFUSERS_AVAILABLE and device.type == 'cuda':
    print('Loading img2img model...')
    img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(device)
    try:
        img2img_pipe.enable_attention_slicing()
    except Exception:
        pass
    print('Img2Img model loaded.')
else:
    img2img_pipe = None
    print('Img2Img requires diffusers + GPU.')

In [ ]:
def img2img_transform(
    image: Image.Image,
    prompt: str,
    strength: float = 0.6,
    num_steps: int = 25,
    seed: int = SEED,
) -> Image.Image | None:
    """Transform an image using text guidance.

    Args:
        image: Input PIL image.
        prompt: Text description guiding the transformation.
        strength: How much to transform (0.0 = none, 1.0 = full).
        num_steps: Number of denoising steps.
        seed: Random seed.

    Returns:
        Transformed PIL Image, or None.
    """
    if img2img_pipe is None:
        print('Img2Img pipeline not available.')
        return None
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    result = img2img_pipe(
        prompt=prompt,
        image=image,
        strength=strength,
        num_inference_steps=num_steps,
        generator=generator,
    )
    
    return result.images[0]


# Demonstrate strength parameter
transform_prompt = 'A watercolor painting of a landscape with mountains'
strengths = [0.3, 0.5, 0.7, 0.9]

if img2img_pipe is not None:
    fig, axes = plt.subplots(1, len(strengths) + 1, figsize=(4 * (len(strengths) + 1), 4))
    axes[0].imshow(test_image)
    axes[0].set_title('Original', fontsize=10)
    axes[0].axis('off')
    
    for idx, strength in enumerate(strengths):
        result = img2img_transform(test_image, transform_prompt, strength=strength)
        if result:
            axes[idx + 1].imshow(result)
            axes[idx + 1].set_title(f'Strength: {strength}', fontsize=10)
            axes[idx + 1].axis('off')
    
    plt.suptitle(f'Img2Img Strength Comparison', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Img2Img strength comparison (would generate with GPU):')
    for s in strengths:
        print(f'  Strength {s}: {"Minimal" if s < 0.4 else "Moderate" if s < 0.7 else "Dramatic"} change')

## Part 3: InstructPix2Pix — Edit with Natural Language

InstructPix2Pix lets you edit images using natural language instructions like "make it winter" or "turn the sky red". Unlike inpainting or img2img, you don't need a mask or a detailed prompt — just a simple instruction.

In [ ]:
if DIFFUSERS_AVAILABLE and device.type == 'cuda':
    print('Loading InstructPix2Pix model...')
    try:
        instruct_pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
            'timbrooks/instruct-pix2pix',
            torch_dtype=torch.float16,
            safety_checker=None,
        ).to(device)
        try:
            instruct_pipe.enable_attention_slicing()
        except Exception:
            pass
        print('InstructPix2Pix model loaded.')
    except Exception as error:
        instruct_pipe = None
        print(f'Could not load InstructPix2Pix: {error}')
else:
    instruct_pipe = None
    print('InstructPix2Pix requires diffusers + GPU.')

In [ ]:
def edit_with_instruction(
    image: Image.Image,
    instruction: str,
    num_steps: int = 20,
    image_guidance_scale: float = 1.5,
    guidance_scale: float = 7.0,
    seed: int = SEED,
) -> Image.Image | None:
    """Edit an image using a natural language instruction.

    Args:
        image: Input PIL image.
        instruction: Editing instruction (e.g., 'make it winter').
        num_steps: Number of denoising steps.
        image_guidance_scale: How much to preserve the original image.
        guidance_scale: How strongly to follow the instruction.
        seed: Random seed.

    Returns:
        Edited PIL Image, or None.
    """
    if instruct_pipe is None:
        print('InstructPix2Pix pipeline not available.')
        return None
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    result = instruct_pipe(
        prompt=instruction,
        image=image,
        num_inference_steps=num_steps,
        image_guidance_scale=image_guidance_scale,
        guidance_scale=guidance_scale,
        generator=generator,
    )
    
    return result.images[0]


# Demonstrate instruction-based editing
instructions = [
    'Make it winter with snow',
    'Turn it into a night scene',
    'Make it look like an oil painting',
]

if instruct_pipe is not None:
    fig, axes = plt.subplots(1, len(instructions) + 1, figsize=(5 * (len(instructions) + 1), 5))
    axes[0].imshow(test_image)
    axes[0].set_title('Original', fontsize=11)
    axes[0].axis('off')
    
    for idx, instruction in enumerate(instructions):
        result = edit_with_instruction(test_image, instruction)
        if result:
            axes[idx + 1].imshow(result)
            axes[idx + 1].set_title(instruction, fontsize=10)
            axes[idx + 1].axis('off')
    
    plt.suptitle('InstructPix2Pix: Natural Language Image Editing', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('InstructPix2Pix examples (would generate with GPU):')
    for inst in instructions:
        print(f'  Instruction: "{inst}"')

## Technique Comparison

Let's summarize the three editing approaches and when to use each.

In [ ]:
comparison_df = pd.DataFrame({
    'Technique': ['Inpainting', 'Img2Img', 'InstructPix2Pix'],
    'Input': ['Image + Mask + Prompt', 'Image + Prompt + Strength', 'Image + Instruction'],
    'Edits': ['Specific masked region', 'Entire image', 'Entire image'],
    'Control': ['High (mask defines area)', 'Medium (strength param)', 'Low (instruction only)'],
    'Best For': [
        'Object removal, region fill',
        'Style transfer, scene changes',
        'Quick edits, natural language',
    ],
    'Model': [
        'SD Inpainting',
        'SD 1.5 (Img2Img mode)',
        'InstructPix2Pix',
    ],
})
comparison_df

## Exercises

1. **Custom masks**: Create different mask shapes (rectangle, freeform, multi-region) for inpainting. How does mask shape affect the result quality?

2. **Img2Img style gallery**: Take one image and transform it into 5 different artistic styles using img2img with strength=0.7.

3. **Instruction chaining**: Apply multiple InstructPix2Pix edits sequentially (output of edit 1 becomes input for edit 2). Does quality degrade?

4. **Real photos**: Use your own photos for editing. Inpaint an object out of a photo, or use InstructPix2Pix to change the season/time of day.

## Key Takeaways

- **Inpainting** fills masked regions with prompted content — ideal for object removal and targeted edits
- **Img2Img** transforms entire images with controllable `strength` (0 = no change, 1 = full regeneration)
- **InstructPix2Pix** enables natural language editing without masks — simplest to use but least precise
- All techniques use the same Stable Diffusion backbone with different conditioning strategies
- GPU with 6GB+ VRAM is required for practical use; memory-efficient attention helps fit larger images

## Next Steps & Resources

**Next notebook**: [04_05 — Document Understanding](04_05_multimodal_document_understanding.ipynb) — extract structured information from documents.

**Resources:**
- [Diffusers Inpainting Guide](https://huggingface.co/docs/diffusers/using-diffusers/inpaint)
- [Img2Img Guide](https://huggingface.co/docs/diffusers/using-diffusers/img2img)
- [InstructPix2Pix Paper](https://arxiv.org/abs/2211.09800) — Learning to Follow Image Editing Instructions
- [ControlNet](https://huggingface.co/docs/diffusers/using-diffusers/controlnet) — Advanced structural control